# Notebook 03 — Zarr Storage

**What you will learn:**
- How Zarr physically stores an array (chunks + metadata files)
- Why chunk size matters for read/write performance
- How to read/write Zarr on local disk and on AWS S3
- The role of `s3fs` as a filesystem abstraction

---

## Background: What is Zarr?

Zarr is a format for storing N-dimensional arrays in **chunks**. Each chunk is a separate file (or S3 object). This means:
- You can read one spatial window without loading the whole array
- Multiple processes can write different chunks in parallel
- The store can live on any filesystem: local disk, S3, GCS, Azure Blob, memory

xarray uses Zarr as its preferred on-disk format because it preserves labelled dimensions, coordinates, and attributes.

In [1]:
import sys
sys.path.insert(0, "..")

import xarray as xr
import zarr
import os
import time
import json
from utils.zarr_helpers import get_s3_store, verify_roundtrip

ZARR_LOCAL = "../data/sentinel2_local.zarr"
ds = xr.open_zarr(ZARR_LOCAL)
print(ds)

<xarray.Dataset> Size: 96MB
Dimensions:      (time: 2, y: 2788, x: 2158)
Coordinates:
  * time         (time) datetime64[ns] 16B 2023-06-04T16:02:09.577000 2023-08...
  * x            (x) float64 17kB 4.355e+05 4.355e+05 ... 4.786e+05 4.786e+05
  * y            (y) float64 22kB 4.428e+06 4.428e+06 ... 4.372e+06 4.372e+06
Data variables:
    blue         (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    nir          (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    spatial_ref  int64 8B ...


## Inspect the Zarr directory structure

A Zarr store is a directory tree. Each array is a sub-directory containing:
- `.zarray` — JSON metadata: dtype, shape, chunk shape, compressor settings
- `.zattrs` — JSON user attributes (coordinates, units, CRS info)
- Numbered files like `0.0`, `0.1` — the compressed chunk data

The root also has `.zmetadata` — a consolidated cache of all metadata.

In [2]:
print("=== Top-level contents ===")
for name in sorted(os.listdir(ZARR_LOCAL)):
    print(" ", name)

print("\n=== Contents of the 'red' array directory ===")
red_dir = os.path.join(ZARR_LOCAL, "red")
for name in sorted(os.listdir(red_dir)):
    full = os.path.join(red_dir, name)
    size = os.path.getsize(full)
    print(f"  {name:30s}  {size:>8,} bytes")

=== Top-level contents ===
  .zattrs
  .zgroup
  .zmetadata
  blue
  green
  nir
  red
  spatial_ref
  time
  x
  y

=== Contents of the 'red' array directory ===
  .zarray                              368 bytes
  .zattrs                              129 bytes
  0.0.0                           6,599,176 bytes
  0.0.1                            455,906 bytes
  0.1.0                           2,325,117 bytes
  0.1.1                            182,085 bytes
  1.0.0                           6,559,913 bytes
  1.0.1                            440,974 bytes
  1.1.0                           2,362,878 bytes
  1.1.1                            177,655 bytes


## Read the .zarray metadata file

`.zarray` is plain JSON. It tells any Zarr reader everything it needs to interpret the chunk files: data type, shape, chunk shape, fill value, and compressor.

In [3]:
zarray_path = os.path.join(ZARR_LOCAL, "red", ".zarray")
with open(zarray_path) as f:
    zarray_meta = json.load(f)

print(json.dumps(zarray_meta, indent=2))

print("\n--- What each field means ---")
print(f"chunks:     {zarray_meta['chunks']}  ← one chunk covers this many elements per dimension")
print(f"dtype:      {zarray_meta['dtype']}")
print(f"shape:      {zarray_meta['shape']}  ← total array shape")
print(f"compressor: {zarray_meta['compressor']}  ← Blosc is the default; lossless")
print(f"fill_value: {zarray_meta['fill_value']}  ← value written for missing data")

{
  "chunks": [
    1,
    2048,
    2048
  ],
  "compressor": {
    "blocksize": 0,
    "clevel": 5,
    "cname": "lz4",
    "id": "blosc",
    "shuffle": 1
  },
  "dtype": "<u2",
  "fill_value": null,
  "filters": null,
  "order": "C",
  "shape": [
    2,
    2788,
    2158
  ],
  "zarr_format": 2
}

--- What each field means ---
chunks:     [1, 2048, 2048]  ← one chunk covers this many elements per dimension
dtype:      <u2
shape:      [2, 2788, 2158]  ← total array shape
compressor: {'blocksize': 0, 'clevel': 5, 'cname': 'lz4', 'id': 'blosc', 'shuffle': 1}  ← Blosc is the default; lossless
fill_value: None  ← value written for missing data


## Verify a roundtrip read

Open the store we wrote in notebook 02 and confirm values match.

In [4]:
ds2 = xr.open_zarr(ZARR_LOCAL)
print("Re-opened dataset:")
print(ds2)

verify_roundtrip(ds, ds2)

# Confirm shapes match
assert ds2["red"].shape == ds["red"].shape
print("Shape matches: ", ds2["red"].shape)

Re-opened dataset:
<xarray.Dataset> Size: 96MB
Dimensions:      (time: 2, y: 2788, x: 2158)
Coordinates:
  * time         (time) datetime64[ns] 16B 2023-06-04T16:02:09.577000 2023-08...
  * x            (x) float64 17kB 4.355e+05 4.355e+05 ... 4.786e+05 4.786e+05
  * y            (y) float64 22kB 4.428e+06 4.428e+06 ... 4.372e+06 4.372e+06
Data variables:
    blue         (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    nir          (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    spatial_ref  int64 8B ...
Roundtrip OK — all variables match.
Shape matches:  (2, 2788, 2158)


## Chunking experiment

Chunk size directly affects both file count and read performance. Re-save with larger spatial chunks and compare.

In [5]:
import time

def store_size_mb(path):
    total = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, dn, fn in os.walk(path)
        for f in fn
    )
    return total / 1e6

ZARR_SMALL_CHUNKS = "../data/sentinel2_smallchunks.zarr"

# Re-chunk to 512×512 (smaller than the original 2048×2048) and write locally.
# Explicitly pass encoding so the new chunk shape overrides the one inherited
# from the source Zarr store — without this xarray's safe-chunks check raises
# a ValueError because the old 2048 encoding conflicts with 512 Dask chunks.
ds_rechunked = ds.chunk({"time": 1, "y": 512, "x": 512})

t0 = time.perf_counter()
ds_rechunked.to_zarr(ZARR_SMALL_CHUNKS, mode="w", encoding={
    v: {"chunks": (1, 512, 512)} for v in ["blue", "green", "red", "nir"]
})
t1 = time.perf_counter()
print(f"Write 1 (512×512 local):  {t1 - t0:.1f}s")

size_orig = store_size_mb(ZARR_LOCAL)
size_small = store_size_mb(ZARR_SMALL_CHUNKS)

print(f"\nOriginal chunks (2048×2048): {size_orig:.1f} MB")
print(f"512×512 chunks:              {size_small:.1f} MB")
print()
print("Count of chunk files (red band):")
for path, label in [(ZARR_LOCAL, "2048×2048"), (ZARR_SMALL_CHUNKS, "512×512")]:
    red_files = [f for f in os.listdir(os.path.join(path, "red")) if not f.startswith(".")]
    print(f"  {label}: {len(red_files)} chunk files")

Write 1 (512×512 local):  9.2s

Original chunks (2048×2048): 79.0 MB
512×512 chunks:              78.1 MB

Count of chunk files (red band):
  2048×2048: 8 chunk files
  512×512: 60 chunk files


## Write to AWS S3

Zarr + s3fs makes writing to S3 as simple as writing to local disk. `s3fs.S3Map` presents S3 as a Python `MutableMapping`, which is all Zarr needs.

Set these environment variables before running:
```
AWS_ACCESS_KEY_ID
AWS_SECRET_ACCESS_KEY
AWS_DEFAULT_REGION
S3_BUCKET          (your bucket name, no s3:// prefix)
```

Local write to data/sentinel2_local.zarr  (36.5s) 2048 x 2048

Local write to data/sentinel2smalechunks.zarr ( 8.84s) 512 x 521

Smaller chunks allow for more parallelism fromDask2

In [6]:
import os
os.environ["S3_BUCKET"] = "jmlane-demo-s3"
S3_BUCKET = os.environ.get("S3_BUCKET", "")

if not S3_BUCKET:
    print("S3_BUCKET not set — skipping S3 write. Set the env var and re-run.")
else:
    import s3fs
    store_s3 = get_s3_store(S3_BUCKET, "sentinel2.zarr")
    print(f"Writing to s3://{S3_BUCKET}/sentinel2.zarr ...")
    t0 = time.perf_counter()
    ds.to_zarr(store_s3, mode="w")
    t1 = time.perf_counter()
    print(f"Write 2 (S3):  {t1 - t0:.1f}s")

Writing to s3://jmlane-demo-s3/sentinel2.zarr ...
Write 2 (S3):  20.8s


## Read back from S3 and verify

In [7]:
if S3_BUCKET:
    store_s3 = get_s3_store(S3_BUCKET, "sentinel2.zarr")
    ds_s3 = xr.open_zarr(store_s3)
    print("Read back from S3:")
    print(ds_s3)
    verify_roundtrip(ds, ds_s3)
else:
    print("Skipping S3 read — S3_BUCKET not set.")

Read back from S3:
<xarray.Dataset> Size: 96MB
Dimensions:      (time: 2, y: 2788, x: 2158)
Coordinates:
  * time         (time) datetime64[ns] 16B 2023-06-04T16:02:09.577000 2023-08...
  * x            (x) float64 17kB 4.355e+05 4.355e+05 ... 4.786e+05 4.786e+05
  * y            (y) float64 22kB 4.428e+06 4.428e+06 ... 4.372e+06 4.372e+06
Data variables:
    blue         (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    green        (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    nir          (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    red          (time, y, x) uint16 24MB dask.array<chunksize=(1, 2048, 2048), meta=np.ndarray>
    spatial_ref  int64 8B ...
Roundtrip OK — all variables match.


## Validation

In [8]:
assert os.path.isdir(ZARR_LOCAL), "Local zarr store missing"
assert os.path.isfile(os.path.join(ZARR_LOCAL, ".zmetadata")), "Missing .zmetadata"
assert os.path.isfile(os.path.join(ZARR_LOCAL, "red", ".zarray")), "Missing red/.zarray"
verify_roundtrip(ds, ds2)
print("All assertions passed.")

Roundtrip OK — all variables match.
All assertions passed.


---
## Learning Checkpoint — Q&A

**Q1:** How does Zarr store a dataset — what is physically written to disk or S3?

> *Each array is stored as a directory of compressed chunk files (named `0.0`, `0.1`, etc.) plus two JSON metadata files: `.zarray` (structure) and `.zattrs` (attributes). The root has `.zmetadata` consolidating all metadata. On S3 each file becomes an object.*

**Q2:** What is a "chunk" in Zarr and why does chunk size matter?

> *A chunk is the unit of I/O — Zarr always reads or writes whole chunks. Small chunks mean many small requests (slow for full-array reads, good for point lookups). Large chunks mean fewer, bigger requests (fast for spatial windows, wasteful for point lookups). Match chunk shape to your access pattern.*

**Q3:** What does `.zarray` contain and why does xarray need it?

> *`.zarray` stores dtype, shape, chunk shape, fill value, and compressor settings. xarray needs it to reconstruct the array dimensions and data type without reading any chunk data — this enables lazy opening of a Zarr store.*

**Q4:** What is the role of `s3fs` — is it a Zarr concept or a filesystem abstraction?

> *s3fs is a filesystem abstraction (not Zarr-specific). It wraps S3 operations behind a Python `MutableMapping` interface. Zarr only knows about abstract key-value stores — it doesn't care whether the backend is disk, S3, or memory, as long as something implements `__getitem__`/`__setitem__`.*